📦 1. Imports
Libraries: requests • BeautifulSoup • pandas • urljoin

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

🌐 2. Connection
Session: Base URL & HTTP session.

In [2]:
base_url = 'https://books.toscrape.com/'
session = requests.Session()

🏠 3. Homepage

Fetch: Pull homepage HTML.

In [4]:
home_response = session.get(base_url)
home_soup = BeautifulSoup(home_response.content, 'html.parser')
home_soup

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s

🏷️ 4. Categories
Extract: All sidebar category links.

In [13]:
category_links = []
categories = home_soup.find('div', class_='side_categories').find('ul').find('ul').find_all('a')
for cat in categories:
    name = cat.text.strip()
    category_url = base_url+ cat['href']
    category_links.append({'name': name, 'url': category_url})
category_links    


[{'name': 'Travel',
  'url': 'https://books.toscrape.com/catalogue/category/books/travel_2/index.html'},
 {'name': 'Mystery',
  'url': 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html'},
 {'name': 'Historical Fiction',
  'url': 'https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html'},
 {'name': 'Sequential Art',
  'url': 'https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html'},
 {'name': 'Classics',
  'url': 'https://books.toscrape.com/catalogue/category/books/classics_6/index.html'},
 {'name': 'Philosophy',
  'url': 'https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html'},
 {'name': 'Romance',
  'url': 'https://books.toscrape.com/catalogue/category/books/romance_8/index.html'},
 {'name': 'Womens Fiction',
  'url': 'https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html'},
 {'name': 'Fiction',
  'url': 'https://books.toscrape.com/catalogue/category/books/ficti

🔄 5. Scrape Loop
Harvest: Extract all books across all pages & categories.


In [31]:
all_books=[]
for cat in category_links:
    category_name = cat['name'].strip()
    url = cat['url']
    print(f"Scraping category: {category_name}...")
    while url:
        response = session.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        books_details = soup.find_all('article',{'class':'product_pod'})
        for book in books_details:
            book_href = book.h3.a['href'].replace('../../../', '')
            img_relative_src = book.find('img')['src'].replace('../../../', '')
            
            all_books.append({
                'category': category_name,
                'title': book.h3.a['title'],
                'price': book.find('p', class_='price_color').text.strip(),
                'availability': book.find('p', class_='instock availability').text.strip(),
                'rating': book.find('p', class_='star-rating')['class'][1],
                'book_url': 'https://books.toscrape.com/catalogue/'+ book_href,       
                'img_src': base_url+ img_relative_src  
            })

        next_btn = soup.find('li', class_='next')
        if next_btn:
            url ='https://books.toscrape.com/catalogue/category/books/mystery_3/'+ next_btn.a['href']
        else:
            url=None

all_books
    

Scraping category: Travel...
Scraping category: Mystery...
Scraping category: Historical Fiction...
Scraping category: Sequential Art...
Scraping category: Classics...
Scraping category: Philosophy...
Scraping category: Romance...
Scraping category: Womens Fiction...
Scraping category: Fiction...
Scraping category: Childrens...
Scraping category: Religion...
Scraping category: Nonfiction...
Scraping category: Music...
Scraping category: Default...
Scraping category: Science Fiction...
Scraping category: Sports and Games...
Scraping category: Add a comment...
Scraping category: Fantasy...
Scraping category: New Adult...
Scraping category: Young Adult...
Scraping category: Science...
Scraping category: Poetry...
Scraping category: Paranormal...
Scraping category: Art...
Scraping category: Psychology...
Scraping category: Autobiography...
Scraping category: Parenting...
Scraping category: Adult Fiction...
Scraping category: Humor...
Scraping category: Horror...
Scraping category: History.

[{'category': 'Travel',
  'title': "It's Only the Himalayas",
  'price': '£45.17',
  'availability': 'In stock',
  'rating': 'Two',
  'book_url': 'https://books.toscrape.com/catalogue/its-only-the-himalayas_981/index.html',
  'img_src': 'https://books.toscrape.com/../media/cache/27/a5/27a53d0bb95bdd88288eaf66c9230d7e.jpg'},
 {'category': 'Travel',
  'title': 'Full Moon over Noah’s Ark: An Odyssey to Mount Ararat and Beyond',
  'price': '£49.43',
  'availability': 'In stock',
  'rating': 'Four',
  'book_url': 'https://books.toscrape.com/catalogue/full-moon-over-noahs-ark-an-odyssey-to-mount-ararat-and-beyond_811/index.html',
  'img_src': 'https://books.toscrape.com/../media/cache/57/77/57770cac1628f4407636635f4b85e88c.jpg'},
 {'category': 'Travel',
  'title': 'See America: A Celebration of Our National Parks & Treasured Sites',
  'price': '£48.87',
  'availability': 'In stock',
  'rating': 'Three',
  'book_url': 'https://books.toscrape.com/catalogue/see-america-a-celebration-of-our-nati

### Creates a DataFrame

In [28]:
df=pd.DataFrame(all_books)
df.head(5)

,category,title,price,availability,rating,book_url,img_src
0,Travel,It's Only the Himalayas,£45.17,In stock,Two,https://books.toscrape.com/catalogue/category/...,https://books.toscrape.com/catalogue/category/...
1,Travel,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,In stock,Four,https://books.toscrape.com/catalogue/category/...,https://books.toscrape.com/catalogue/category/...
2,Travel,See America: A Celebration of Our National Par...,£48.87,In stock,Three,https://books.toscrape.com/catalogue/category/...,https://books.toscrape.com/catalogue/category/...
3,Travel,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,In stock,Two,https://books.toscrape.com/catalogue/category/...,https://books.toscrape.com/catalogue/category/...
4,Travel,Under the Tuscan Sun,£37.33,In stock,Three,https://books.toscrape.com/catalogue/category/...,https://books.toscrape.com/catalogue/category/...


### 📊To export the data to csv file

In [ ]:
df.to_csv('all_books_dataset.csv', index=False)